# 04 — Modelling (next-day AQI, 2023–2025 data)
**Input:** `data/Cleaned/aqi_2023_25_features.csv` (from notebook 03)

| Task | Target | Models | Metrics |
|---|---|---|---|
| A. Classification | tomorrow's `AQI_Bucket` | Logistic Regression, Random Forest, Hist Gradient Boosting | per-class precision/recall/F1, macro-F1, confusion matrix |
| B. Regression | tomorrow's `AQI` | Ridge, Random Forest, Hist Gradient Boosting | MAE, RMSE, R² |

## Four rules that make this valid
1. **Split by time, not randomly.** Train up to Oct 2024, test Nov 2024 – Apr 2025. The test period contains a full winter, the hardest and most important case. A random split would let neighbouring days leak between train and test.
2. **Baselines first.** "Tomorrow = today" (persistence) is strong. A model that does not beat it is not useful.
3. **Class imbalance:** use balanced class weights and judge by **macro-F1** (it treats rare classes like Severe equally). Accuracy alone would hide failure on rare classes.
4. **Tune with `TimeSeriesSplit`**, never shuffled folds, and make sure the rows are sorted by **date** first (shown below).

Runtime: on the full data (~240k rows) the Random Forests take a few minutes. Set `SAMPLE_FRAC` below 1 to try the notebook quickly.

In [1]:
import sys, os, time, warnings
sys.path.append("..")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import (RandomForestClassifier, RandomForestRegressor,
                              HistGradientBoostingClassifier, HistGradientBoostingRegressor)
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             classification_report, ConfusionMatrixDisplay,
                             mean_absolute_error, mean_squared_error, r2_score)

from src.aqi_common import CLASS_ORDER, TRAIN_END, TEST_START, ALL_FEATURES, build_preprocessor

sns.set_style("whitegrid")
FIG = "../reports/figures/2023_25"
for d in [FIG, "../models", "../reports"]:
    os.makedirs(d, exist_ok=True)

def save(name):
    plt.tight_layout()
    plt.savefig(f"{FIG}/2023_25_{name}.png", dpi=150, bbox_inches="tight")
    plt.show()

SAMPLE_FRAC = 1.0      # use e.g. 0.3 for a quick trial run
RUN_TUNING = True      # set False to skip the grid searches
MERGE_SEVERE = False   # True merges 'Very Poor' and 'Severe' (see the class balance below)
FEATURES = ALL_FEATURES

## Step 1 — Load and split by time
Training rows are those with `Date < 2024-10-31`; test rows have `Date >= 2024-11-01`. The days in between are left out on purpose: a row dated Oct 31 has its *label* (Nov 1) inside the test period, so keeping it would leak.

In [2]:
df = pd.read_csv("../data/Cleaned/aqi_2023_25_features.csv", parse_dates=['Date'])

def maybe_merge(s):
    return s.replace({'Very Poor': 'Very Poor/Severe', 'Severe': 'Very Poor/Severe'}) if MERGE_SEVERE else s

df['y_class'] = maybe_merge(df['target_bucket_next'])
df['today_class'] = maybe_merge(df['AQI_Bucket'])
base_order = ['Good', 'Satisfactory', 'Moderate', 'Poor', 'Very Poor/Severe'] if MERGE_SEVERE else CLASS_ORDER
labels = [c for c in base_order if c in set(df['y_class'])]

train = df[df['Date'] < TRAIN_END].copy()
test = df[df['Date'] >= TEST_START].copy()
if SAMPLE_FRAC < 1:
    train = train.sample(frac=SAMPLE_FRAC, random_state=0)

# sort by DATE (all cities together) - required for TimeSeriesSplit to cut along time
train = train.sort_values(['Date', 'City']).reset_index(drop=True)
test = test.sort_values(['Date', 'City']).reset_index(drop=True)

X_train, X_test = train[FEATURES], test[FEATURES]
print("Train:", len(train), train['Date'].min().date(), "->", train['Date'].max().date())
print("Test :", len(test), test['Date'].min().date(), "->", test['Date'].max().date())

dist = pd.concat([train['y_class'].value_counts(normalize=True).mul(100).round(1).rename('train %'),
                  test['y_class'].value_counts(normalize=True).mul(100).round(1).rename('test %')], axis=1)
dist.reindex(labels)

Train: 141433 2022-04-10 -> 2024-10-30
Test : 31826 2024-11-01 -> 2025-04-29


,train %,test %
y_class,,
Good,18.7,7.9
Satisfactory,38.2,33.0
Moderate,31.3,45.9
Poor,8.9,11.4
Very Poor,2.7,1.7
Severe,0.2,0.1


**Check the class balance above.** If *Severe* is a tiny share, a model cannot learn it. In that case set `MERGE_SEVERE = True`, rerun from the top, and state the choice in your report. The test period (winter) will also have fewer *Good* days than training, so the test set is harder.

## Step 2 — Metric helpers

In [3]:
def clf_metrics(y_true, y_pred, name):
    return {'Model': name,
            'Accuracy': accuracy_score(y_true, y_pred),
            'MacroF1': f1_score(y_true, y_pred, labels=labels, average='macro', zero_division=0),
            'MacroPrecision': precision_score(y_true, y_pred, labels=labels, average='macro', zero_division=0),
            'MacroRecall': recall_score(y_true, y_pred, labels=labels, average='macro', zero_division=0)}

def reg_metrics(y_true, y_pred, name):
    return {'Model': name,
            'MAE': mean_absolute_error(y_true, y_pred),
            'RMSE': float(np.sqrt(mean_squared_error(y_true, y_pred))),
            'R2': r2_score(y_true, y_pred)}

---
# Part A — Classification: tomorrow's AQI category

## Step 3 — Baselines
- *Always the most common class* (a floor).
- *Persistence:* tomorrow's category = today's category.

**Takeaway:** *(persistence accuracy is the number to beat; note its macro-F1 too.)*

## Step 4 — Train the three classifiers
Each model is a `Pipeline` (preprocessing + model), so the same transformations are applied to train and test, and the city statistics are learned from training rows only.

**Takeaway:** *(does any model beat persistence on macro-F1? On accuracy? They can disagree, which is why we report both.)*

## Step 5 — Tune the gradient boosting model (time-ordered CV)
`TimeSeriesSplit` trains on earlier dates and validates on later ones. This only works because `train` is sorted by date.

## Step 6 — Per-class evaluation of the best model
Precision = of the days we flagged as a class, how many really were. Recall = of the days that really were that class, how many we caught. For *Severe* and *Very Poor*, recall is the number that matters for public health.

*Caution: "best" is picked using the test period, which slightly flatters it. In the report, say so, or choose using the cross-validation scores from the tuning step.*

## Step 7 — Confusion matrix (best model vs persistence)
Rows are the true category, columns the prediction, each row scaled to sum to 1. A good model has a strong diagonal, and its mistakes sit **next to** the diagonal (Poor mistaken for Very Poor is mild; Good mistaken for Severe would be serious).

**Takeaway:** *(which categories are hardest? Are the errors between neighbouring categories?)*

---
# Part B — Regression: tomorrow's AQI value

## Step 8 — Baselines
- *Persistence:* tomorrow = today.
- *7-day mean:* tomorrow = the average of the last 7 days.

## Step 9 — Train the three regressors

## Step 10 — Tune gradient boosting (time-ordered CV)

**Takeaway:** *(by how many AQI points does the best model beat persistence? MAE is in AQI units, so this is easy to explain.)*

## Step 11 — Predicted vs actual

## Step 12 — One city over the test winter

## Step 13 — Save results, predictions and models
The analysis notebook (05) reads these files. Model files (`*.pkl`) are in `.gitignore`, so teammates regenerate them by running this notebook.

## Summary 
- A results table for each task: baselines vs three models (`results_*_2023_25.csv`).
- A clear statement of whether the models beat persistence, and by how much, on a **test winter**.
- The confusion matrix and the per-class report for the best classifier.
- Limitations: only about 2.3 years of data, a single test winter, no pollutant concentrations, and the Diwali effect is based on two festivals.


